# Feature Engineering: 모델 학습용 피처셋 및 분할 데이터 구축

EDA 단계의 핵심 인사이트와 모델 설계 전략을 반영하여 데이터 누수(Data Leakage) 없는 후보 피처 보존, 점진적 실험용 Feature Sets 정의, Group-aware 데이터 분할 수행

## 0. 필수 라이브러리 및 경로 설정

In [12]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 경로 설정
CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ('notebooks', 'test') else CURRENT_DIR
INPUT_DIR = ROOT / 'results' / 'eda' / 'tables'
OUTPUT_DIR = ROOT / 'results' / 'features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'입력 데이터 경로: {INPUT_DIR}')
print(f'피처 출력 경로: {OUTPUT_DIR}')

입력 데이터 경로: /Users/yujin.je/Workspace/ds-mini/results/eda/tables
피처 출력 경로: /Users/yujin.je/Workspace/ds-mini/results/features


## 1. 데이터 로드 및 유효 표본 필터링
- 타겟 결측 셀(10개: Batch 2 8개, Batch 3 2개)은 EOL 미도달 또는 센서 결측으로 지도학습 대상에서 제외
- 결측 타겟의 인위적 대치 없이 유효 셀 129개(Batch 1 46개, Batch 2 39개, Batch 3 44개)만 추출

In [13]:
# EDA 집계 피처 로드
df = pd.read_csv(INPUT_DIR / 'all_batch_cell_features.csv')

# 타겟 결측치 제거 및 유효 표본 필터링
valid_df = df[df.cycle_life.notna()].copy().reset_index(drop=True)

print(f'전체 셀 수: {len(df)}개 | 유효 셀 수: {len(valid_df)}개 (결측 {len(df) - len(valid_df)}개 제외)')
display(valid_df.groupby('batch').agg(
    total=('cell_id', 'count'),
    median_life=('cycle_life', 'median'),
    min_life=('cycle_life', 'min'),
    max_life=('cycle_life', 'max')
))

전체 셀 수: 139개 | 유효 셀 수: 129개 (결측 10개 제외)


,total,median_life,min_life,max_life
batch,,,,
Batch 1,46,858.5,534.0,1227.0
Batch 2,39,472.0,392.0,1186.0
Batch 3,44,1005.5,541.0,1935.0


## 2. 피처 생성 및 Feature Sets 정의
- **핵심 피처 (`log_delta_q_var`)**: 3개 배치 전반에서 가장 강력한 음의 상관(-0.7 이하)을 보인 $\Delta Q(V)$ 분산에 $\log_{10}$ 변환을 적용하여 극단적 왜도 완화
- **저장용 후보 피처 보존 (`CANDIDATE_FEATURES`)**: 모델링 단계에서 다양한 가설 검증과 점진적 실험(Ablation)이 가능하도록 후보 피처 풀 전체를 CSV에 보존 (CSV 저장 $\neq$ 모델 동시 투입)
- **점진적 실험용 Feature Sets (A/B/C/D/ALT)**:
  - **Set A (단독 기준)**: `log_delta_q_var`
  - **Set B (운전조건 보정)**: Set A + 충전조건 연속형 수치 3종 (`first_C_rate`, `switch_SOC`, `second_C_rate`)
  - **Set C (물리신호 확장)**: Set B + 초기 100 사이클 물리 보조 3종 (`mean_chargetime`, `mean_IR`, `mean_Tavg`)
  - **Set D (용량 변동성 추가)**: Set C + 방전용량 변동성 (`std_QD`)
  - **ALT_delta_q_min (대체 실험)**: `delta_q_min` 단독 투입 (다중공선성 방지를 위해 Set A와 개별 비교)
  - **E_mean_QD (별도 실험)**: Set D + 초기 방전용량 평균 (`mean_QD`)
- **제외 피처**: 미래 정보 누수인 `knee_cycle` 및 사이클 100 이후 통계 완전 배제, 다중공선성 변수인 `mean_Tmax` 제외, Batch 1 내 분산이 없는 `newstructure` 제외

In [14]:
# 1) 핵심 피처 log10 변환
valid_df['log_delta_q_var'] = np.log10(valid_df['delta_q_var'])

# 2) 저장용 후보 피처 풀 정의
META_COLS = ['batch', 'cell_id', 'charging_policy', 'cycle_life']
CANDIDATE_FEATURES = [
    'log_delta_q_var',
    'delta_q_min',
    'first_C_rate',
    'switch_SOC',
    'second_C_rate',
    'mean_chargetime',
    'mean_IR',
    'mean_Tavg',
    'std_QD',
    'mean_QD',
]

# 3) 모델링 실험용 Feature Sets 정의
FEATURE_SETS = {
    'A': ['log_delta_q_var'],
    'B': ['log_delta_q_var', 'first_C_rate', 'switch_SOC', 'second_C_rate'],
    'C': ['log_delta_q_var', 'first_C_rate', 'switch_SOC', 'second_C_rate', 'mean_chargetime', 'mean_IR', 'mean_Tavg'],
    'D': ['log_delta_q_var', 'first_C_rate', 'switch_SOC', 'second_C_rate', 'mean_chargetime', 'mean_IR', 'mean_Tavg', 'std_QD'],
    'ALT_delta_q_min': ['delta_q_min'],
    'E_mean_QD': ['log_delta_q_var', 'first_C_rate', 'switch_SOC', 'second_C_rate', 'mean_chargetime', 'mean_IR', 'mean_Tavg', 'std_QD', 'mean_QD'],
}

# 4) 피처 결측치 검증
missing = valid_df[CANDIDATE_FEATURES].isna().sum()
display(missing[missing > 0])
assert missing.sum() == 0, '피처 결측치가 존재합니다. 03_modeling Pipeline에서 imputation 필요'
print(f'후보 피처 {len(CANDIDATE_FEATURES)}개 전체 결측치 0개 검증 완료')

for set_name, cols in FEATURE_SETS.items():
    print(f'Set {set_name:16s} ({len(cols)}개): {cols}')

Series([], dtype: int64)

후보 피처 10개 전체 결측치 0개 검증 완료
Set A                (1개): ['log_delta_q_var']
Set B                (4개): ['log_delta_q_var', 'first_C_rate', 'switch_SOC', 'second_C_rate']
Set C                (7개): ['log_delta_q_var', 'first_C_rate', 'switch_SOC', 'second_C_rate', 'mean_chargetime', 'mean_IR', 'mean_Tavg']
Set D                (8개): ['log_delta_q_var', 'first_C_rate', 'switch_SOC', 'second_C_rate', 'mean_chargetime', 'mean_IR', 'mean_Tavg', 'std_QD']
Set ALT_delta_q_min  (1개): ['delta_q_min']
Set E_mean_QD        (9개): ['log_delta_q_var', 'first_C_rate', 'switch_SOC', 'second_C_rate', 'mean_chargetime', 'mean_IR', 'mean_Tavg', 'std_QD', 'mean_QD']


## 3. 누수 없는 데이터셋 분할 (Data Splitting)
- **Train / Hold-out Valid 분할 (Batch 1)**: 동일 충전 프로토콜 셀이 Train과 Valid에 동시 투입되는 정보 누수를 차단하기 위해 `charging_policy` 기준 `GroupShuffleSplit` 적용 (Train 35개, Valid 11개)
- **Test 1 (Batch 2)**: 단수명 중심 환경(중앙값 472 cycle)에서의 필수 외부 일반화 검증 데이터셋 (39개)
- **Test 2 (Batch 3)**: 장수명 중심 환경(중앙값 1,005.5 cycle)에서의 추가 외부 일반화 검증 데이터셋 (44개)

In [15]:
# Batch 분리
b1 = valid_df[valid_df.batch == 'Batch 1'].copy().reset_index(drop=True)
test_b2 = valid_df[valid_df.batch == 'Batch 2'].copy().reset_index(drop=True)
test_b3 = valid_df[valid_df.batch == 'Batch 3'].copy().reset_index(drop=True)

# Batch 1 내부 GroupShuffleSplit (charging_policy 기준)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(b1, groups=b1['charging_policy']))

train_df = b1.iloc[train_idx].copy().reset_index(drop=True)
val_df = b1.iloc[val_idx].copy().reset_index(drop=True)

# 분할 결과 검증
policy_overlap = set(train_df.charging_policy).intersection(set(val_df.charging_policy))
assert len(policy_overlap) == 0, 'Train과 Valid 간 정책 중복 발생'

print(f'Train 세트 (Batch 1): {len(train_df)}개 셀 ({train_df.charging_policy.nunique()}개 정책) | 중앙 수명: {train_df.cycle_life.median():.1f}')
print(f'Valid 세트 (Batch 1): {len(val_df)}개 셀 ({val_df.charging_policy.nunique()}개 정책) | 중앙 수명: {val_df.cycle_life.median():.1f}')
print(f'Test 1 세트 (Batch 2): {len(test_b2)}개 셀 ({test_b2.charging_policy.nunique()}개 정책) | 중앙 수명: {test_b2.cycle_life.median():.1f}')
print(f'Test 2 세트 (Batch 3): {len(test_b3)}개 셀 ({test_b3.charging_policy.nunique()}개 정책) | 중앙 수명: {test_b3.cycle_life.median():.1f}')

Train 세트 (Batch 1): 35개 셀 (18개 정책) | 중앙 수명: 857.0
Valid 세트 (Batch 1): 11개 셀 (5개 정책) | 중앙 수명: 860.0
Test 1 세트 (Batch 2): 39개 셀 (12개 정책) | 중앙 수명: 472.0
Test 2 세트 (Batch 3): 44개 셀 (8개 정책) | 중앙 수명: 1005.5


## 4. 데이터셋 저장 및 모델링 연계
- **FE 스케일링 배제 원칙**: Feature Engineering 단계에서는 표준화(`fit`)를 수행하지 않고 원본(Raw) 상태로만 저장
- **03 Modeling 파이프라인 연계**: 표준화 및 결측 대치는 03_modeling에서 `GroupKFold` 내부 `Pipeline(StandardScaler)`을 통해 Fold 단위로 수행하여 데이터 누수 원천 차단
- **저장 산출물**: Train, Valid, Test 1, Test 2 원본 CSV 4종 및 모델링용 `feature_sets.json` 저장

In [16]:
# 1) Raw 데이터셋 구성
all_cols = META_COLS + CANDIDATE_FEATURES
train_raw = train_df[all_cols].copy()
val_raw = val_df[all_cols].copy()
test_b2_raw = test_b2[all_cols].copy()
test_b3_raw = test_b3[all_cols].copy()

# 2) Raw CSV 4개 저장
datasets = {
    'train_features_raw.csv': train_raw,
    'valid_features_raw.csv': val_raw,
    'test_b2_features_raw.csv': test_b2_raw,
    'test_b3_features_raw.csv': test_b3_raw,
}

for filename, dataset in datasets.items():
    dataset.to_csv(OUTPUT_DIR / filename, index=False)

# 3) Feature Sets 메타데이터 JSON 저장 (03_modeling 연계용)
with open(OUTPUT_DIR / 'feature_sets.json', 'w', encoding='utf-8') as f:
    json.dump(FEATURE_SETS, f, ensure_ascii=False, indent=2)

# 4) 최종 무결성 검증
assert len(train_raw) == 35
assert len(val_raw) == 11
assert len(test_b2_raw) == 39
assert len(test_b3_raw) == 44
assert (OUTPUT_DIR / 'feature_sets.json').exists()
assert len(list(OUTPUT_DIR.glob('*.csv'))) == 4
print('4개 분할 데이터셋 및 feature_sets.json 저장 완료 (무결성 검증 통과)')

4개 분할 데이터셋 및 feature_sets.json 저장 완료 (무결성 검증 통과)
